# v11 C (GPU 0): FREEZE, then SILENCE, while training the next phase (count-matched)
- **freeze** (the paper's second test): the chosen neurons keep working but their incoming weights cannot change.
- **silence**: the chosen neurons are held at their average for the whole training.
- Conditions: self@1.0 vs task@1.0 (the untouched runs come from notebook B; any missing ones are run here). Same 12 starts, 3 seeds.
Results: `analysis/v11/train_next/` (same folder as B) · progress: `tail -f logs/v11_C_freeze_silence_next.log`

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "0"; LOG = "v11_C_freeze_silence_next"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
FILES, HF_REPO = [], None
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
if not FILES:
    print("could not list the HF repo: only checkpoints already on this machine will be used")

def have(run, ph):
    return bool(glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"))

def get(run, phases):
    """download what is missing; returns the phases that are on this machine afterwards"""
    ok = []
    for ph in phases:
        if not have(run, ph):
            name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
            if name:
                try:
                    hf_hub_download(HF_REPO, name[0], local_dir="runs")
                except Exception as e:
                    print("download failed:", run, ph, type(e).__name__, e)
        if have(run, ph):
            ok.append(ph)
        else:
            print("not available (not here, not on HF):", run, ph)
    return ok

def run(langset, *cmd):
    """one call of core_v11.py; output shown here AND appended to logs/<LOG>.log (watch with: tail -f logs/<LOG>.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", "core_v11.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f"logs/{LOG}.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

def summary(part, out=None):
    cmd = [sys.executable, "core_v11.py", "--summarize", "--part", part] + (["--out", out] if out else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-12000:]); print(r.stderr[-3000:])

DIST, HIN, MIX, REL = "v4b_continual_s0", "v4b_only_hi_s0", "v6b_mixed_s0", "v8_indic_continual_s0"

In [ ]:
STARTS = [("distant", "distant", DIST, p) for p in [10, 11, 12, 40, 41, 42]] + [("indic", "related", REL, p) for p in [10, 11, 12, 13, 14, 15]]

for mode in ["freeze", "silence"]:
    for langset, fam, r, ph in STARTS:
        if get(r, [ph]):
            run(langset, "--part", "train", "--task", "next", "--mode", mode, "--conds", "none,self@1.0,task@1.0", "--family", fam,
                "--model", f"{fam}{ph}={r}:{ph}", "--seeds", "0,1,2")
    summary("train", "train_next")

In [ ]:
summary("train", "train_next")
if os.path.exists("analysis/v11/train_next/curves.png"): display(Image("analysis/v11/train_next/curves.png"))